# CropCop Track B — R07 Internal Preflight Surface

**Not a claim-run notebook.** The only supported protected Track-B execution surface is `trackb_r07_master.ipynb`.

This notebook exists for inspection and frozen-input/environment preflight only. It intentionally cannot start external R07 inference because protected execution requires the master's durable attempt ledger, source acquisition, scratch isolation, private evidence round-trip, and terminal publication controls.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/trackb_r07_preflight')
SCRATCH_ROOT = Path('/kaggle/tmp/cropcop_trackb_r07_preflight')

manifests = []
for p in sorted(INPUT_ROOT.glob('**/TRACKB_INPUT_MANIFEST.json')):
    obj = json.loads(p.read_text(encoding='utf-8'))
    manifests.append((obj.get('role'), p, obj))
roles = [r for r, _, _ in manifests]
required = {'core', 'historical_compare', 'gvlid_v5', 'irish_potato'}
if set(roles) != required:
    raise RuntimeError(f'Expected exactly {sorted(required)}, got {sorted(roles)}')

core_path, core = next((p, o) for r, p, o in manifests if r == 'core')
repo_rel = str(core.get('repository_root', '')).strip()
if not repo_rel:
    raise RuntimeError('core TRACKB_INPUT_MANIFEST.json must define repository_root')
REPO_ROOT = (core_path.parent / repo_rel).resolve()
RUNNER = REPO_ROOT / 'journal_extension/scripts/run_trackb_r07.py'
if not RUNNER.is_file():
    raise RuntimeError(f'Track-B runner missing: {RUNNER}')

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
if SCRATCH_ROOT.exists():
    shutil.rmtree(SCRATCH_ROOT)

print('Track-B input roles:', roles)
print('Repository root:', REPO_ROOT)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)


In [ ]:
cmd = [
    sys.executable, str(RUNNER),
    '--input-root', str(INPUT_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
    '--device', 'cuda:0',
    '--workers', '4',
    '--mode', 'preflight',
]
print('Launching prediction-free preflight:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

gate = json.loads((OUTPUT_ROOT / 'PREFLIGHT_PASS.json').read_text(encoding='utf-8'))
if gate.get('status') != 'PASS':
    raise RuntimeError('Track-B preflight did not PASS')
print(json.dumps(gate, indent=2, sort_keys=True))


## Boundary

A PASS here proves only the frozen authority/environment/R07-family replay and input-package preflight. It does **not** authorize manuscript claims or protected external inference. Use the master notebook for the controlled claim run.
